# Decision-Based Agent

This notebook demonstrates an agentic workflow where an AI agent intelligently decides which tool to use based on the input query.

## Overview

The agent will:
1. Receive a user query
2. Analyze the query to determine the best tool
3. Execute the chosen tool
4. Return the result

## Architecture

- **Agent**: Decision-making component that selects appropriate tools
- **Tools**: Various capabilities (calculator, search, data analysis, etc.)
- **LLM**: Language model for understanding queries and making decisions

## Setup and Imports

In [ ]:
import json
import re
from typing import Dict, List, Any, Callable
from datetime import datetime
import math

# For LLM integration (using Anthropic Claude as example)
# Uncomment and install if you want to use real LLM:
# !pip install anthropic
# import anthropic

## Define Tools

Each tool has a specific capability that the agent can use.

In [ ]:
class Tool:
    """Base class for all tools"""
    def __init__(self, name: str, description: str):
        self.name = name
        self.description = description
    
    def execute(self, *args, **kwargs) -> Any:
        raise NotImplementedError("Each tool must implement execute method")
    
    def to_dict(self) -> Dict:
        return {
            "name": self.name,
            "description": self.description
        }

In [ ]:
class CalculatorTool(Tool):
    """Performs mathematical calculations"""
    def __init__(self):
        super().__init__(
            name="calculator",
            description="Performs mathematical calculations. Use for arithmetic, algebra, or mathematical operations."
        )
    
    def execute(self, expression: str) -> Dict[str, Any]:
        try:
            # Safe evaluation of mathematical expressions
            allowed_names = {k: v for k, v in math.__dict__.items() if not k.startswith("__")}
            result = eval(expression, {"__builtins__": {}}, allowed_names)
            return {
                "success": True,
                "result": result,
                "expression": expression
            }
        except Exception as e:
            return {
                "success": False,
                "error": str(e),
                "expression": expression
            }

In [ ]:
class SearchTool(Tool):
    """Simulates web search functionality"""
    def __init__(self):
        super().__init__(
            name="search",
            description="Searches for information on the web. Use for finding facts, current events, or general knowledge."
        )
        # Simulated search database
        self.search_db = {
            "weather": "The current weather is sunny with a temperature of 72°F.",
            "python": "Python is a high-level programming language known for its simplicity and readability.",
            "ai": "Artificial Intelligence (AI) is the simulation of human intelligence by machines.",
            "machine learning": "Machine learning is a subset of AI that enables systems to learn from data.",
        }
    
    def execute(self, query: str) -> Dict[str, Any]:
        query_lower = query.lower()
        # Simple keyword matching
        for key, value in self.search_db.items():
            if key in query_lower:
                return {
                    "success": True,
                    "query": query,
                    "results": value
                }
        return {
            "success": True,
            "query": query,
            "results": f"Search results for '{query}': [Simulated search - no specific results found]"
        }

In [ ]:
class DataAnalysisTool(Tool):
    """Analyzes data and provides statistics"""
    def __init__(self):
        super().__init__(
            name="data_analysis",
            description="Analyzes numerical data and provides statistics like mean, median, sum, etc."
        )
    
    def execute(self, data: List[float]) -> Dict[str, Any]:
        try:
            if not data:
                return {"success": False, "error": "No data provided"}
            
            sorted_data = sorted(data)
            n = len(data)
            
            analysis = {
                "success": True,
                "count": n,
                "sum": sum(data),
                "mean": sum(data) / n,
                "min": min(data),
                "max": max(data),
                "median": sorted_data[n // 2] if n % 2 == 1 else (sorted_data[n // 2 - 1] + sorted_data[n // 2]) / 2,
                "range": max(data) - min(data)
            }
            return analysis
        except Exception as e:
            return {"success": False, "error": str(e)}

In [ ]:
class DateTimeTool(Tool):
    """Provides date and time information"""
    def __init__(self):
        super().__init__(
            name="datetime",
            description="Gets current date, time, or performs date calculations."
        )
    
    def execute(self, operation: str = "current") -> Dict[str, Any]:
        try:
            now = datetime.now()
            
            if operation == "current":
                return {
                    "success": True,
                    "datetime": now.strftime("%Y-%m-%d %H:%M:%S"),
                    "date": now.strftime("%Y-%m-%d"),
                    "time": now.strftime("%H:%M:%S"),
                    "day_of_week": now.strftime("%A")
                }
            else:
                return {"success": False, "error": "Unknown operation"}
        except Exception as e:
            return {"success": False, "error": str(e)}

In [ ]:
class TextAnalysisTool(Tool):
    """Analyzes text for various properties"""
    def __init__(self):
        super().__init__(
            name="text_analysis",
            description="Analyzes text for word count, character count, and other text properties."
        )
    
    def execute(self, text: str) -> Dict[str, Any]:
        try:
            words = text.split()
            sentences = re.split(r'[.!?]+', text)
            sentences = [s for s in sentences if s.strip()]
            
            return {
                "success": True,
                "character_count": len(text),
                "word_count": len(words),
                "sentence_count": len(sentences),
                "average_word_length": sum(len(word) for word in words) / len(words) if words else 0
            }
        except Exception as e:
            return {"success": False, "error": str(e)}

## Decision-Based Agent

The agent analyzes queries and selects the most appropriate tool.

In [ ]:
class DecisionAgent:
    """Agent that decides which tool to use based on the query"""
    
    def __init__(self, tools: List[Tool], use_llm: bool = False, api_key: str = None):
        self.tools = {tool.name: tool for tool in tools}
        self.use_llm = use_llm
        self.api_key = api_key
        self.history = []
    
    def _decide_with_rules(self, query: str) -> str:
        """Rule-based decision making (fallback method)"""
        query_lower = query.lower()
        
        # Mathematical keywords
        if any(keyword in query_lower for keyword in 
               ['calculate', 'compute', 'add', 'subtract', 'multiply', 'divide', 
                'sum', 'difference', 'product', 'quotient', '+', '-', '*', '/', 
                'square', 'sqrt', 'power', 'equation']):
            return 'calculator'
        
        # Search keywords
        if any(keyword in query_lower for keyword in 
               ['search', 'find', 'look up', 'what is', 'who is', 'when', 
                'where', 'weather', 'information about']):
            return 'search'
        
        # Data analysis keywords
        if any(keyword in query_lower for keyword in 
               ['analyze', 'statistics', 'mean', 'median', 'average', 'data', 
                'numbers', 'min', 'max', 'standard deviation']):
            return 'data_analysis'
        
        # DateTime keywords
        if any(keyword in query_lower for keyword in 
               ['date', 'time', 'today', 'now', 'current', 'day', 'hour']):
            return 'datetime'
        
        # Text analysis keywords
        if any(keyword in query_lower for keyword in 
               ['count words', 'count characters', 'text', 'analyze text', 
                'word count', 'character count']):
            return 'text_analysis'
        
        # Default to search
        return 'search'
    
    def _decide_with_llm(self, query: str) -> str:
        """LLM-based decision making (advanced method)"""
        # This is a placeholder for LLM integration
        # Uncomment and modify if you have API access:
        """
        client = anthropic.Anthropic(api_key=self.api_key)
        
        tool_descriptions = "\n".join(
            f"- {name}: {tool.description}" 
            for name, tool in self.tools.items()
        )
        
        prompt = f"""Given the following tools:
{tool_descriptions}

User query: {query}

Which tool should be used? Respond with ONLY the tool name."""
        
        message = client.messages.create(
            model="claude-3-5-sonnet-20241022",
            max_tokens=50,
            messages=[{"role": "user", "content": prompt}]
        )
        
        tool_name = message.content[0].text.strip().lower()
        return tool_name if tool_name in self.tools else 'search'
        """
        
        # Fallback to rule-based for demo
        return self._decide_with_rules(query)
    
    def process_query(self, query: str, **kwargs) -> Dict[str, Any]:
        """Main method to process a query"""
        print(f"\n{'='*60}")
        print(f"Query: {query}")
        print(f"{'='*60}")
        
        # Step 1: Decide which tool to use
        if self.use_llm and self.api_key:
            selected_tool_name = self._decide_with_llm(query)
        else:
            selected_tool_name = self._decide_with_rules(query)
        
        print(f"\n🤖 Agent Decision: Using '{selected_tool_name}' tool")
        
        # Step 2: Execute the tool
        if selected_tool_name not in self.tools:
            result = {
                "success": False,
                "error": f"Tool '{selected_tool_name}' not found"
            }
        else:
            tool = self.tools[selected_tool_name]
            result = tool.execute(**kwargs)
        
        # Step 3: Store in history and return
        self.history.append({
            "query": query,
            "tool_used": selected_tool_name,
            "result": result,
            "timestamp": datetime.now().isoformat()
        })
        
        print(f"\n📊 Result:")
        print(json.dumps(result, indent=2))
        
        return {
            "query": query,
            "tool_used": selected_tool_name,
            "result": result
        }
    
    def get_history(self) -> List[Dict[str, Any]]:
        """Returns the agent's execution history"""
        return self.history

## Initialize Agent and Tools

In [ ]:
# Create tool instances
tools = [
    CalculatorTool(),
    SearchTool(),
    DataAnalysisTool(),
    DateTimeTool(),
    TextAnalysisTool()
]

# Initialize agent (using rule-based decision making)
agent = DecisionAgent(tools, use_llm=False)

print("✅ Agent initialized with the following tools:")
for tool in tools:
    print(f"  - {tool.name}: {tool.description}")

## Example 1: Mathematical Calculation

In [ ]:
# The agent should recognize this needs the calculator
result = agent.process_query(
    "Calculate the square root of 144",
    expression="sqrt(144)"
)

## Example 2: Information Search

In [ ]:
# The agent should recognize this needs the search tool
result = agent.process_query(
    "What is Python programming language?",
    query="Python programming language"
)

## Example 3: Data Analysis

In [ ]:
# The agent should recognize this needs data analysis
result = agent.process_query(
    "Analyze the statistics of these numbers",
    data=[10, 20, 30, 40, 50, 25, 35, 45]
)

## Example 4: Date and Time

In [ ]:
# The agent should recognize this needs datetime tool
result = agent.process_query(
    "What is the current date and time?",
    operation="current"
)

## Example 5: Text Analysis

In [ ]:
# The agent should recognize this needs text analysis
sample_text = "Artificial intelligence is transforming the world. Machine learning enables computers to learn from data. This is an exciting time for technology!"

result = agent.process_query(
    "Count the words and characters in this text",
    text=sample_text
)

## View Agent History

In [ ]:
# Display execution history
print("\n" + "="*60)
print("AGENT EXECUTION HISTORY")
print("="*60)

history = agent.get_history()
for i, entry in enumerate(history, 1):
    print(f"\n[{i}] Query: {entry['query']}")
    print(f"    Tool Used: {entry['tool_used']}")
    print(f"    Timestamp: {entry['timestamp']}")
    print(f"    Success: {entry['result'].get('success', False)}")

## Interactive Demo

Try your own queries!

In [ ]:
def interactive_agent():
    """Interactive mode for testing the agent"""
    print("\n" + "="*60)
    print("INTERACTIVE DECISION-BASED AGENT")
    print("="*60)
    print("\nAvailable commands:")
    print("  - Type your query and press Enter")
    print("  - Type 'history' to see execution history")
    print("  - Type 'quit' to exit")
    print("\nNote: Depending on your query, you may need to provide additional parameters.")
    print("      This is a simplified demo - the agent will use sensible defaults.\n")
    
    while True:
        user_input = input("\nYour query: ").strip()
        
        if user_input.lower() == 'quit':
            print("Goodbye!")
            break
        
        if user_input.lower() == 'history':
            history = agent.get_history()
            if not history:
                print("No history yet.")
            else:
                for i, entry in enumerate(history, 1):
                    print(f"\n[{i}] {entry['query']} -> {entry['tool_used']}")
            continue
        
        if not user_input:
            continue
        
        # Simple parameter inference
        query_lower = user_input.lower()
        
        if any(kw in query_lower for kw in ['calculate', 'compute', '+', '-', '*', '/']):
            # Try to extract mathematical expression
            # For demo, use a default if we can't extract
            expr = user_input.split('calculate')[-1].strip() if 'calculate' in query_lower else "2+2"
            agent.process_query(user_input, expression=expr)
        
        elif 'analyze' in query_lower and 'data' in query_lower:
            # Use sample data
            agent.process_query(user_input, data=[1, 2, 3, 4, 5])
        
        elif 'text' in query_lower or 'word' in query_lower or 'character' in query_lower:
            # Use the query itself as text to analyze
            agent.process_query(user_input, text=user_input)
        
        elif 'date' in query_lower or 'time' in query_lower:
            agent.process_query(user_input, operation="current")
        
        else:
            # Default to search
            agent.process_query(user_input, query=user_input)

# Uncomment to run interactive mode:
# interactive_agent()

## Advanced: Using LLM for Decision Making

To use an LLM (like Claude) for more intelligent decision making:

1. Install the Anthropic SDK: `pip install anthropic`
2. Get your API key from [console.anthropic.com](https://console.anthropic.com)
3. Uncomment the LLM integration code in `_decide_with_llm` method
4. Create agent with: `agent = DecisionAgent(tools, use_llm=True, api_key="your-api-key")`

In [ ]:
# Example with LLM (requires API key)
# agent_with_llm = DecisionAgent(tools, use_llm=True, api_key="your-api-key-here")
# result = agent_with_llm.process_query("What's 15% of 200?", expression="0.15 * 200")

## Summary

This notebook demonstrates a decision-based agent that:

1. **Analyzes user queries** to understand intent
2. **Selects appropriate tools** from a toolkit
3. **Executes the chosen tool** with relevant parameters
4. **Returns results** and maintains execution history

### Key Components:

- **Tools**: Modular, reusable capabilities (calculator, search, data analysis, etc.)
- **Decision Logic**: Rule-based or LLM-based tool selection
- **Agent**: Orchestrates the workflow and manages state

### Extending the Agent:

You can easily add new tools by:
1. Creating a new class that inherits from `Tool`
2. Implementing the `execute` method
3. Adding it to the tools list
4. Updating the decision logic (or let the LLM handle it)

This pattern is fundamental to building more complex agentic systems!